# 📊 Notebook 06b — Análisis de alpha en EMA Dialog2Flow · `version_4`

Esta notebook consolida los resultados de EMA y genera visualizaciones para responder:

> ¿Cuánta historia conviene codificar para el retriever?

Se analizan curvas `alpha vs performance` usando métricas ANN, geométricas y semántico-funcionales.

## 1️⃣ Imports y rutas

In [ ]:
!pip install -q pandas numpy matplotlib seaborn

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = Path("/content/drive/MyDrive/ANN/TF")
VERSION = "version_4"

RESULTS_DIR = BASE_DIR / "resultados" / VERSION
BENCHMARK_RESULTS_DIR = RESULTS_DIR / "benchmark_ann"
GEOMETRY_RESULTS_DIR = RESULTS_DIR / "geometry"
SEMANTIC_RESULTS_DIR = RESULTS_DIR / "semantic_llm"
FIGURES_DIR = RESULTS_DIR / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ALPHA_VALUES = [round(x / 10, 1) for x in range(1, 10)]

def variant_alpha(v):
    v = str(v)
    if v.startswith("ema_alpha_"):
        return float(v.replace("ema_alpha_", "").replace("_", "."))
    return np.nan

## 2️⃣ Carga de resultados

In [ ]:
benchmark_path = BENCHMARK_RESULTS_DIR / "resultados_ann_ema_dialog2flow_latest.csv"
geometry_basic_path = GEOMETRY_RESULTS_DIR / "geometry_ema_basic_dialog2flow_latest.csv"
geometry_overlap_path = GEOMETRY_RESULTS_DIR / "geometry_ema_overlap_dialog2flow_latest.csv"
semantic_summary_path = SEMANTIC_RESULTS_DIR / "llm_semantic_summary_ema_dialog2flow_latest.csv"
semantic_comparison_path = SEMANTIC_RESULTS_DIR / "ema_comparison_vs_baselines_dialog2flow_latest.csv"

df_bench = pd.read_csv(benchmark_path) if benchmark_path.exists() else pd.DataFrame()
df_geo = pd.read_csv(geometry_basic_path) if geometry_basic_path.exists() else pd.DataFrame()
df_overlap = pd.read_csv(geometry_overlap_path) if geometry_overlap_path.exists() else pd.DataFrame()
df_sem = pd.read_csv(semantic_summary_path) if semantic_summary_path.exists() else pd.DataFrame()
df_comp = pd.read_csv(semantic_comparison_path) if semantic_comparison_path.exists() else pd.DataFrame()

print("Benchmark:", df_bench.shape)
print("Geometry basic:", df_geo.shape)
print("Geometry overlap:", df_overlap.shape)
print("Semantic summary:", df_sem.shape)
print("Comparison:", df_comp.shape)

## 3️⃣ ANN: alpha vs Recall/QPS

In [ ]:
if not df_bench.empty:
    df_ema = df_bench[df_bench["variant"].astype(str).str.startswith("ema_alpha_")].copy()
    df_ema["alpha"] = df_ema["variant"].map(variant_alpha)

    # Configuraciones representativas
    df_rep = df_ema[
        (
            (df_ema["index_type"] == "IVF") & (df_ema["nprobe"] == 64)
        ) |
        (
            (df_ema["index_type"] == "HNSW") & (df_ema["efSearch"] == 256)
        ) |
        (
            (df_ema["index_type"] == "IVFPQ") & (df_ema["nprobe"] == 64)
        )
    ].copy()

    display(df_rep[["variant", "alpha", "index_type", "recall@10", "recall@100", "qps", "memory_mb"]])

    for metric in ["recall@10", "recall@100", "qps"]:
        plt.figure(figsize=(8, 5))
        for index_type, g in df_rep.groupby("index_type"):
            g = g.sort_values("alpha")
            plt.plot(g["alpha"], g[metric], marker="o", label=index_type)
        plt.xlabel("alpha")
        plt.ylabel(metric)
        plt.title(f"EMA Dialog2Flow — alpha vs {metric}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        out = FIGURES_DIR / f"ema_dialog2flow_alpha_vs_{metric.replace('@','at')}.pdf"
        plt.savefig(out, bbox_inches="tight")
        plt.show()
        print("Guardado:", out)
else:
    print("No se encontró benchmark latest.")

## 4️⃣ Geometría: alpha vs similitud y overlap

In [ ]:
if not df_geo.empty:
    plt.figure(figsize=(8, 5))
    g = df_geo.sort_values("alpha")
    plt.plot(g["alpha"], g["similarity_mean"], marker="o", label="Similitud media")
    plt.plot(g["alpha"], g["similarity_median"], marker="o", label="Similitud mediana")
    plt.xlabel("alpha")
    plt.ylabel("Similitud coseno estático-EMA")
    plt.title("EMA Dialog2Flow — alpha vs similitud con embedding estático")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out = FIGURES_DIR / "ema_dialog2flow_alpha_vs_static_similarity.pdf"
    plt.savefig(out, bbox_inches="tight")
    plt.show()
    print("Guardado:", out)

if not df_overlap.empty:
    plt.figure(figsize=(8, 5))
    g = df_overlap.sort_values("alpha")
    for metric in ["overlap@1", "overlap@10", "overlap@100"]:
        plt.plot(g["alpha"], g[metric], marker="o", label=metric)
    plt.xlabel("alpha")
    plt.ylabel("Overlap exacto estático vs EMA")
    plt.title("EMA Dialog2Flow — alpha vs overlap de vecindades")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out = FIGURES_DIR / "ema_dialog2flow_alpha_vs_overlap.pdf"
    plt.savefig(out, bbox_inches="tight")
    plt.show()
    print("Guardado:", out)

## 5️⃣ Semántica: alpha vs MSS@10

In [ ]:
if not df_sem.empty:
    df_sem_ema = df_sem[df_sem["variant"].astype(str).str.startswith("ema_alpha_")].copy()
    df_sem_ema["alpha"] = df_sem_ema["variant"].map(variant_alpha)

    display(df_sem_ema[[
        "variant", "alpha", "index_type", "n_queries",
        "mean_semantic_score_global_at10", "sd_semantic_score_global_at10"
    ]])

    plt.figure(figsize=(8, 5))
    for index_type, g in df_sem_ema.groupby("index_type"):
        g = g.sort_values("alpha")
        plt.plot(g["alpha"], g["mean_semantic_score_global_at10"], marker="o", label=index_type)

    # Líneas de referencia si están disponibles
    for baseline in ["estatico", "acumulativo_uniforme"]:
        b = df_sem[df_sem["variant"] == baseline]
        if not b.empty:
            ref = b.groupby("index_type")["mean_semantic_score_global_at10"].mean().mean()
            plt.axhline(ref, linestyle="--", alpha=0.5, label=f"Promedio {baseline}")

    plt.xlabel("alpha")
    plt.ylabel("MSS@10")
    plt.title("EMA Dialog2Flow — alpha vs MSS@10")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out = FIGURES_DIR / "ema_dialog2flow_alpha_vs_mss_at10.pdf"
    plt.savefig(out, bbox_inches="tight")
    plt.show()
    print("Guardado:", out)
else:
    print("No se encontró summary semántico latest.")

## 6️⃣ Comparación pareada

In [ ]:
if not df_comp.empty:
    display(df_comp.sort_values(["index_type", "alpha"]))

    df_comp_ema = df_comp[df_comp["variant"].astype(str).str.startswith("ema_alpha_")].copy()

    for metric in ["mean_delta_vs_static", "pct_variant_better_than_static", "mean_delta_vs_accumulative_uniform", "pct_variant_better_than_accumulative_uniform"]:
        if metric not in df_comp_ema.columns:
            continue
        plt.figure(figsize=(8, 5))
        for index_type, g in df_comp_ema.groupby("index_type"):
            g = g.sort_values("alpha")
            plt.plot(g["alpha"], g[metric], marker="o", label=index_type)
        plt.xlabel("alpha")
        plt.ylabel(metric)
        plt.title(f"EMA Dialog2Flow — alpha vs {metric}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        out = FIGURES_DIR / f"ema_dialog2flow_alpha_vs_{metric}.pdf"
        plt.savefig(out, bbox_inches="tight")
        plt.show()
        print("Guardado:", out)
else:
    print("No se encontró comparación pareada latest.")

## 7️⃣ Tabla resumen por alpha

In [ ]:
# Resumen compacto combinando ANN representativo y semántica, si ambos existen.
if not df_bench.empty:
    df_rep_ann = df_bench.copy()
    df_rep_ann["alpha"] = df_rep_ann["variant"].map(variant_alpha)
    df_rep_ann = df_rep_ann[df_rep_ann["variant"].astype(str).str.startswith("ema_alpha_")]
    df_rep_ann = df_rep_ann[
        ((df_rep_ann["index_type"] == "IVF") & (df_rep_ann["nprobe"] == 64)) |
        ((df_rep_ann["index_type"] == "HNSW") & (df_rep_ann["efSearch"] == 256)) |
        ((df_rep_ann["index_type"] == "IVFPQ") & (df_rep_ann["nprobe"] == 64))
    ][["variant", "alpha", "index_type", "recall@10", "recall@100", "qps"]]

    if not df_sem.empty:
        df_sem_small = df_sem[df_sem["variant"].astype(str).str.startswith("ema_alpha_")][[
            "variant", "index_type", "mean_semantic_score_global_at10", "sd_semantic_score_global_at10"
        ]]
        summary = df_rep_ann.merge(df_sem_small, on=["variant", "index_type"], how="left")
    else:
        summary = df_rep_ann

    display(summary.sort_values(["index_type", "alpha"]))

    summary_path = RESULTS_DIR / "ema_dialog2flow_alpha_summary_latest.csv"
    summary.to_csv(summary_path, index=False)
    print("Guardado:", summary_path)